# Download and process WHACS offshore waves

Download WHACS bulk and partition spectra, remap them onto metamodel goals, and
write the processed zarr used by dynamic forcing and hindcast reconstruction.

**Prerequisites:** ``04_Select_Goals_by_Hindcasts.ipynb`` (``<Case>_Goals.json``).
**Next:** ``07b_Satellite_Calibration.ipynb`` validates (and calibrates) this
hindcast against satellite altimetry.

## Configuration


In [1]:
import numpy as np
import xarray as xr

from utils.config import case_name, get_offshore_wave, load_goals, resolve

whacs_data = xr.open_dataset(
    resolve(f"inputs/WHACS/processed/{case_name()}_WHACS_seapoint.zarr")
).load()
whacs_data = whacs_data.assign({"tp": 1.0 / whacs_data["fp"]})
whacs_data["tp"].attrs = {
    "long_name": "Wave peak period",
    "units": "s",
}
whacs_data

<xarray.Dataset> Size: 2GB
Dimensions:                      (time: 394464, seapoint: 16)
Coordinates:
  * time                         (time) datetime64[ns] 3MB 1979-01-01 ... 202...
  * seapoint                     (seapoint) int32 64B 970691 971044 ... 970352
Data variables: (12/42)
    cge                          (time, seapoint) float64 50MB 23.4 ... 10.0
    dir                          (time, seapoint) float64 50MB 348.1 ... 328.9
    dp                           (time, seapoint) float64 50MB 348.0 ... 298.0
    fp                           (time, seapoint) float64 50MB 0.061 ... 0.073
    hs                           (time, seapoint) float64 50MB 2.16 ... 1.388
    latitude                     (time, seapoint) float32 25MB -13.22 ... -13.84
    ...                           ...
    spr                          (time, seapoint) float64 50MB 51.4 ... 76.7
    t01                          (time, seapoint) float64 50MB 8.89 ... 8.91
    t02                          (time, seapoint) float64 50MB 7.68 7.67 ... 7.6
    uwnd                         (time, seapoint) float64 50MB -2.5 ... -0.6
    vwnd                         (time, seapoint) float64 50MB -4.3 ... -1.2
    tp                           (time, seapoint) float64 50MB 16.39 ... 13.7

In [2]:
# Offshore node per goal, matched by lon/lat to the goals JSON snap points
# (hindcast.whacs.longitude/latitude). Some hindcasts are gridded rather
# than point-organized, so matching by position is more robust than by id.
goals = load_goals()
seapoint_lon = whacs_data["longitude"].isel(time=-1)
seapoint_lat = whacs_data["latitude"].isel(time=-1)
ordered_seapoints_for_goals = []
for goal in goals.values():
    snap = goal["hindcast"]["whacs"]
    distance = np.hypot(
        seapoint_lon - snap["longitude"], seapoint_lat - snap["latitude"]
    )
    ordered_seapoints_for_goals.append(
        int(whacs_data["seapoint"][int(distance.argmin())])
    )

whacs_data = whacs_data.sel(seapoint=ordered_seapoints_for_goals)[
    [
        "hs",
        "tp",
        "dir",
        "spr",
        "phs0",
        "ptp0",
        "pdir0",
        "pspr0",
        "phs1",
        "ptp1",
        "pdir1",
        "pspr1",
        "phs2",
        "ptp2",
        "pdir2",
        "pspr2",
        "phs3",
        "ptp3",
        "pdir3",
        "pspr3",
    ]
].rename({"seapoint": "goal"})
whacs_data["goal"] = [int(k) for k in goals]
whacs_data = whacs_data.assign_coords(
    {"seapoint": ("goal", ordered_seapoints_for_goals)}
)

Bulk `tp = 1 / fp` explodes when the spectral peak falls below the WHACS
frequency grid (hours with Tp of 50-250 s). Those take the peak period of the
most energetic partition (`clean_peak_periods`); partition periods above
`max_tp` become NaN.

In [3]:
from bluemath_tk.waves.hywaves.offshore import clean_peak_periods, scale_partitions_to_bulk

max_tp = 25.0  # s, largest valid peak period
whacs_data = clean_peak_periods(whacs_data, max_tp=max_tp)
print(f"bulk tp replaced: {whacs_data.attrs['tp_replaced']}, partition tp removed: {whacs_data.attrs['ptp_removed']}")

bulk tp replaced: 28, partition tp removed: 0


**Partition energy.** WHACS keeps the wind sea and three swells; where the
sea state has more systems (multimodal tropical seas) the partitions hold less
energy than the bulk spectrum, and the spectral reconstruction (a sum of
partitions) would start low. `scale_partitions_to_bulk` gives each hour's
partitions the energy they miss, in proportion to theirs (`phs` x
`hs / sqrt(sum phs²)`; periods, directions and spreads kept). The share they
held before, per goal, is printed below. The satellite calibration (07b) scales
the bulk and the partitions by the same factors, so it keeps the closure.

In [4]:
energy = sum(whacs_data[f"phs{i}"].fillna(0.0) ** 2 for i in range(4))
captured = (energy.sum("time") / (whacs_data["hs"] ** 2).sum("time")).to_series()
print("partition energy / bulk energy per goal, before scaling:")
print(captured.round(3).to_string())

whacs_data = scale_partitions_to_bulk(whacs_data)
whacs_data.to_zarr(resolve(get_offshore_wave("WHACS")["data_path"]), mode="w")

partition energy / bulk energy per goal, before scaling:
goal
1     0.871
2     0.886
3     0.946
4     0.936
5     0.944
6     0.891
7     0.905
8     0.903
9     0.903
10    0.906
11    0.908
12    0.909
13    0.940
14    0.938
15    0.934
16    0.896


In [5]:
xr.open_zarr(resolve(get_offshore_wave("WHACS")["data_path"])).load()

<xarray.Dataset> Size: 1GB
Dimensions:   (time: 394464, goal: 16)
Coordinates:
  * time      (time) datetime64[ns] 3MB 1979-01-01 ... 2023-12-31T23:00:00
  * goal      (goal) int64 128B 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16
    seapoint  (goal) int64 128B 970691 971044 971049 ... 969892 970007 970352
Data variables: (12/20)
    dir       (time, goal) float64 50MB 348.1 348.8 347.7 ... 184.5 191.0 328.9
    hs        (time, goal) float64 50MB 2.16 2.236 2.26 ... 1.088 1.128 1.388
    pdir0     (time, goal) float64 50MB nan nan nan nan nan ... nan nan nan nan
    pdir1     (time, goal) float64 50MB 344.5 342.4 311.8 ... 176.0 170.8 352.5
    pdir2     (time, goal) float64 50MB 20.1 19.7 20.2 ... 220.1 219.8 167.3
    pdir3     (time, goal) float64 50MB 313.8 315.3 347.8 ... 189.2 188.9 216.3
    ...        ...
    ptp0      (time, goal) float64 50MB nan nan nan nan nan ... nan nan nan nan
    ptp1      (time, goal) float64 50MB 16.53 16.48 8.47 ... 8.87 8.8 13.48
    ptp2      (time, goal) float64 50MB 10.05 12.81 10.0 ... 13.82 13.81 8.7
    ptp3      (time, goal) float64 50MB 8.49 8.46 16.78 ... 14.64 14.63 13.85
    spr       (time, goal) float64 50MB 51.4 50.4 53.0 52.1 ... 52.4 58.9 76.7
    tp        (time, goal) float64 50MB 16.39 16.39 16.39 ... 14.08 13.89 13.7
Attributes:
    tp_replaced:             28
    ptp_removed:             0
    partition_energy_share:  0.9118